# Facebook Engagement Analysis – Data Cleaning & Preprocessing

Author: Nick Peril  
Project: TKH Phase I Portfolio

### Objective
To clean and preprocess the dataset in preparation for modeling.

In [1]:
import pandas as pd
import numpy as np

### Loading my dataset:

In [2]:
df = pd.read_csv("/Users/saadult/facebook-metrics/data/01_raw/dataset_Facebook.csv", sep=';')
df.head(10)

,Page total likes,Type,Category,Post Month,Post Weekday,Post Hour,Paid,Lifetime Post Total Reach,Lifetime Post Total Impressions,Lifetime Engaged Users,Lifetime Post Consumers,Lifetime Post Consumptions,Lifetime Post Impressions by people who have liked your Page,Lifetime Post reach by people who like your Page,Lifetime People who have liked your Page and engaged with your post,comment,like,share,Total Interactions
0,139441,Photo,2,12,4,3,0.0,2752,5091,178,109,159,3078,1640,119,4,79.0,17.0,100
1,139441,Status,2,12,3,10,0.0,10460,19057,1457,1361,1674,11710,6112,1108,5,130.0,29.0,164
2,139441,Photo,3,12,3,3,0.0,2413,4373,177,113,154,2812,1503,132,0,66.0,14.0,80
3,139441,Photo,2,12,2,10,1.0,50128,87991,2211,790,1119,61027,32048,1386,58,1572.0,147.0,1777
4,139441,Photo,2,12,2,3,0.0,7244,13594,671,410,580,6228,3200,396,19,325.0,49.0,393
5,139441,Status,2,12,1,9,0.0,10472,20849,1191,1073,1389,16034,7852,1016,1,152.0,33.0,186
6,139441,Photo,3,12,1,3,1.0,11692,19479,481,265,364,15432,9328,379,3,249.0,27.0,279
7,139441,Photo,3,12,7,9,1.0,13720,24137,537,232,305,19728,11056,422,0,325.0,14.0,339
8,139441,Status,2,12,7,3,0.0,11844,22538,1530,1407,1692,15220,7912,1250,0,161.0,31.0,192
9,139441,Photo,3,12,6,10,0.0,4694,8668,280,183,250,4309,2324,199,3,113.0,26.0,142


### Checking for missing values:

In [3]:
df.isnull().sum()

Page total likes                                                       0
Type                                                                   0
Category                                                               0
Post Month                                                             0
Post Weekday                                                           0
Post Hour                                                              0
Paid                                                                   1
Lifetime Post Total Reach                                              0
Lifetime Post Total Impressions                                        0
Lifetime Engaged Users                                                 0
Lifetime Post Consumers                                                0
Lifetime Post Consumptions                                             0
Lifetime Post Impressions by people who have liked your Page           0
Lifetime Post reach by people who like your Page   

There are missing values under the following columns: 'Paid', 'like', and 'share'

### Handling Missing Values

There are missing values in few columns such as 'paid', 'like', and 'share'. Since the missing values only accounts to less than 1% of the total dataset, these rows were removed to avoid artificial values through imputation.

In [4]:
df = df.dropna()

# Feature Engineering

### Cleaning Column Names

In [5]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [6]:
df.columns

Index(['page_total_likes', 'type', 'category', 'post_month', 'post_weekday',
       'post_hour', 'paid', 'lifetime_post_total_reach',
       'lifetime_post_total_impressions', 'lifetime_engaged_users',
       'lifetime_post_consumers', 'lifetime_post_consumptions',
       'lifetime_post_impressions_by_people_who_have_liked_your_page',
       'lifetime_post_reach_by_people_who_like_your_page',
       'lifetime_people_who_have_liked_your_page_and_engaged_with_your_post',
       'comment', 'like', 'share', 'total_interactions'],
      dtype='object')

Since my datasets' column names have spaces and capitalizations, I cleaned them following the naming conventions for variables for easier coding.

### Verify total interactions consistency

In [7]:
df['calculated_total'] = df['like'] + df['share'] + df['comment']

In [8]:
(df['calculated_total'] == df['total_interactions']).all()

np.True_

Since total_interactions variable is already available, I just need to confirm it's consistency by creating a new variable called 'calculated_total' by getting the sum of like, share, and comment. I confirmed this by having a boolean statement to see if it is equal to 'total_interactions.

### Creating Engagement Rate

Raw engagement could be misleading. A post that reached 100,000 people and got 500 likes is weaker than a post that reached 2,000 people and got 300 likes. So I am going to make an engagement rate variable to measure engagement relative to exposure.

Engagement Rate = Total Interactions / Reach

In [9]:
df['engagement_rate'] = df['total_interactions'] / df['lifetime_post_total_reach']

### Handling categorical variables

In the Facebook Metrics dataset, there are categorical variables under the 'type' column. Since Linear Regression only works with numbers, I have to convert them into numerical form, and I will be using the One-Hot Encoding method.

In [10]:
df.select_dtypes(include='object')

,type
0,Photo
1,Status
2,Photo
3,Photo
4,Photo
...,...
494,Photo
495,Photo
496,Photo
497,Photo


In [11]:
df = pd.get_dummies(df, columns=['type'], drop_first=True, dtype=int)

In [12]:
df.head()

,page_total_likes,category,post_month,post_weekday,post_hour,paid,lifetime_post_total_reach,lifetime_post_total_impressions,lifetime_engaged_users,lifetime_post_consumers,...,lifetime_people_who_have_liked_your_page_and_engaged_with_your_post,comment,like,share,total_interactions,calculated_total,engagement_rate,type_Photo,type_Status,type_Video
0,139441,2,12,4,3,0.0,2752,5091,178,109,...,119,4,79.0,17.0,100,100.0,0.036337,1,0,0
1,139441,2,12,3,10,0.0,10460,19057,1457,1361,...,1108,5,130.0,29.0,164,164.0,0.015679,0,1,0
2,139441,3,12,3,3,0.0,2413,4373,177,113,...,132,0,66.0,14.0,80,80.0,0.033154,1,0,0
3,139441,2,12,2,10,1.0,50128,87991,2211,790,...,1386,58,1572.0,147.0,1777,1777.0,0.035449,1,0,0
4,139441,2,12,2,3,0.0,7244,13594,671,410,...,396,19,325.0,49.0,393,393.0,0.054252,1,0,0


The pd.get_dummies() created three new columns -- 'type_Photo', 'type_Status', and 'type_Video' to identify the type of post. If all 3 are False or 0, it means type is photo.

In [13]:
df.select_dtypes(include='object')

""
0
1
2
3
4
...
494
495
496
497
